In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("sqlite:///../../mlflow.db")
mlflow.set_experiment("sentinel_delivery_anomaly")  

df = pd.read_parquet("../../data/processed/delivery_features.parquet")
IF_FEATURES = ['duration_robust_city','distance_robust_city','implied_speed_kmh',
               'accept_hour','is_instant_delivery','is_ghost_dispatch','aoi_type']
X = df[IF_FEATURES].values
print(f"Feature matrix: {X.shape}")

params = {
    'n_estimators': 100,
    'max_samples': 256,
    'contamination': 0.01,
    'random_state': 42,
}

with mlflow.start_run(run_name="isoforest_contam_01"):
    iso = IsolationForest(**params, n_jobs=-1)
    iso.fit(X)
    df['anomaly_score'] = iso.decision_function(X)
    df['is_anomaly'] = (iso.predict(X) == -1).astype(int)

    n_anom = int(df['is_anomaly'].sum())
    mlflow.log_params(params)
    mlflow.log_param("n_features", len(IF_FEATURES))
    mlflow.log_param("features", ", ".join(IF_FEATURES))
    mlflow.log_metric("n_anomalies", n_anom)
    mlflow.log_metric("anomaly_rate", df['is_anomaly'].mean())
    mlflow.log_metric("score_min", df['anomaly_score'].min())
    mlflow.log_metric("score_mean", df['anomaly_score'].mean())
    mlflow.set_tags({"model": "isolation_forest", "stage": "baseline"})

    print(f"Anomalies flagged: {n_anom:,} ({df['is_anomaly'].mean()*100:.2f}%)")
    print(f"\nScore distribution:")
    print(df['anomaly_score'].describe().to_string())

/opt/anaconda3/envs/sentinel/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/17 17:59:52 INFO mlflow.tracking.fluent: Experiment with name 'sentinel_delivery_anomaly' does not exist. Creating a new experiment.


Feature matrix: (4512573, 7)
Anomalies flagged: 45,122 (1.00%)

Score distribution:
count    4.512573e+06
mean     2.288524e-01
std      6.145917e-02
min     -1.526918e-01
25%      2.018951e-01
50%      2.456807e-01
75%      2.761333e-01
max      2.936526e-01


In [ ]:
anom = df[df['is_anomaly']==1]
normal = df[df['is_anomaly']==0]

print("=== ANOMALIES vs NORMAL (median values) ===")
compare = pd.DataFrame({
    'anomaly_median': [anom['delivery_duration_min'].median(), anom['distance_km'].median(),
                       anom['implied_speed_kmh'].median(), anom['accept_hour'].median(),
                       anom['is_instant_delivery'].mean(), anom['is_ghost_dispatch'].mean()],
    'normal_median': [normal['delivery_duration_min'].median(), normal['distance_km'].median(),
                      normal['implied_speed_kmh'].median(), normal['accept_hour'].median(),
                      normal['is_instant_delivery'].mean(), normal['is_ghost_dispatch'].mean()],
}, index=['duration_min','distance_km','speed_kmh','accept_hour','pct_instant','pct_ghost'])
print(compare.round(3).to_string())

print("\n=== Did it catch the obvious anomalies? ===")
print(f"Instant deliveries flagged: {anom['is_instant_delivery'].sum():,} / {df['is_instant_delivery'].sum():,} ({anom['is_instant_delivery'].sum()/df['is_instant_delivery'].sum()*100:.1f}%)")
print(f"Ghost dispatches flagged:   {anom['is_ghost_dispatch'].sum():,} / {df['is_ghost_dispatch'].sum():,} ({anom['is_ghost_dispatch'].sum()/df['is_ghost_dispatch'].sum()*100:.1f}%)")
print(f"Speed >100km/h flagged:     {(anom['implied_speed_kmh']>100).sum():,} / {(df['implied_speed_kmh']>100).sum():,}")

print("\n=== TOP 10 MOST ANOMALOUS ===")
top = df.nsmallest(10, 'anomaly_score')[['city','delivery_duration_min','distance_km',
        'implied_speed_kmh','accept_hour','is_instant_delivery','is_ghost_dispatch','anomaly_score']]
print(top.round(2).to_string())

=== ANOMALIES vs NORMAL (median values) ===
              anomaly_median  normal_median
duration_min         103.000        105.000
distance_km           16.530          1.924
speed_kmh             13.884          1.142
accept_hour           12.000         11.000
pct_instant            0.012          0.003
pct_ghost              0.001          0.001

=== Did it catch the obvious anomalies? ===
Instant deliveries flagged: 545 / 12,930 (4.2%)
Ghost dispatches flagged:   36 / 3,375 (1.1%)
Speed >100km/h flagged:     9,179 / 13,018

=== TOP 10 MOST ANOMALOUS ===
              city  delivery_duration_min  distance_km  implied_speed_kmh  accept_hour  is_instant_delivery  is_ghost_dispatch  anomaly_score
868289    Shanghai                    1.0       361.81           21708.64           23                    0                  0          -0.15
871372    Shanghai                    3.0       215.62            4312.49           21                    0                  0          -0.15
3862511  

In [ ]:
IF_FEATURES = ['duration_robust_city','distance_robust_city','implied_speed_kmh',
               'accept_hour','is_instant_delivery','is_ghost_dispatch','aoi_type']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[IF_FEATURES].values)

with mlflow.start_run(run_name="isoforest_scaled_contam_01"):
    iso_b = IsolationForest(n_estimators=100, max_samples=256,
                            contamination=0.01, random_state=42, n_jobs=-1)
    iso_b.fit(X_scaled)
    df['anomaly_score_b'] = iso_b.decision_function(X_scaled)
    df['is_anomaly_b'] = (iso_b.predict(X_scaled) == -1).astype(int)

    mlflow.log_params({'n_estimators':100,'max_samples':256,'contamination':0.01,'scaled':True})
    mlflow.log_metric("n_anomalies", int(df['is_anomaly_b'].sum()))
    mlflow.set_tags({"model":"isolation_forest","stage":"scaled_optionB"})

print("=== CATCH RATES: Option A vs Option B ===")
for name, col in [("A (unscaled)", "is_anomaly"), ("B (scaled)", "is_anomaly_b")]:
    a = df[df[col]==1]
    inst = a['is_instant_delivery'].sum() / df['is_instant_delivery'].sum() * 100
    ghost = a['is_ghost_dispatch'].sum() / df['is_ghost_dispatch'].sum() * 100
    fast = (a['implied_speed_kmh']>100).sum() / (df['implied_speed_kmh']>100).sum() * 100
    print(f"\n{name}: {df[col].sum():,} flagged")
    print(f"  instant caught: {inst:.1f}%  |  ghost caught: {ghost:.1f}%  |  speed>100 caught: {fast:.1f}%")

overlap = ((df['is_anomaly']==1) & (df['is_anomaly_b']==1)).sum()
print(f"\nOverlap (both flag): {overlap:,}")
print(f"Only A flags: {((df['is_anomaly']==1)&(df['is_anomaly_b']==0)).sum():,}")
print(f"Only B flags: {((df['is_anomaly']==0)&(df['is_anomaly_b']==1)).sum():,}")

print("\n=== TOP 10 MOST ANOMALOUS (Option B scaled) ===")
top_b = df.nsmallest(10, 'anomaly_score_b')[['city','delivery_duration_min','distance_km',
        'implied_speed_kmh','accept_hour','is_instant_delivery','is_ghost_dispatch']]
print(top_b.round(2).to_string())

=== CATCH RATES: Option A vs Option B ===

A (unscaled): 45,122 flagged
  instant caught: 4.2%  |  ghost caught: 1.1%  |  speed>100 caught: 70.5%

B (scaled): 45,122 flagged
  instant caught: 4.2%  |  ghost caught: 1.1%  |  speed>100 caught: 70.5%

Overlap (both flag): 45,122
Only A flags: 0
Only B flags: 0

=== TOP 10 MOST ANOMALOUS (Option B scaled) ===
              city  delivery_duration_min  distance_km  implied_speed_kmh  accept_hour  is_instant_delivery  is_ghost_dispatch
868289    Shanghai                    1.0       361.81           21708.64           23                    0                  0
871372    Shanghai                    3.0       215.62            4312.49           21                    0                  0
3862511  Chongqing                   19.0        85.48             269.93           21                    0                  0
2694789   Hangzhou                    9.0        46.64             310.91           21                    0                  0
871369 

In [ ]:
df['rule_instant']          = df['is_instant_delivery'] == 1
df['rule_ghost']            = df['is_ghost_dispatch'] == 1
df['rule_impossible_speed'] = df['implied_speed_kmh'] > 100
df['rule_extreme_duration'] = df['delivery_duration_min'] > df.groupby('city')['delivery_duration_min'].transform(lambda x: x.quantile(0.999))

df['final_anomaly'] = (
    (df['is_anomaly'] == 1) |
    df['rule_instant'] | df['rule_ghost'] |
    df['rule_impossible_speed'] | df['rule_extreme_duration']
).astype(int)

df['anomaly_reason'] = ''
df.loc[df['rule_impossible_speed'], 'anomaly_reason'] += 'impossible_speed, '
df.loc[df['rule_instant'],          'anomaly_reason'] += 'instant_delivery, '
df.loc[df['rule_ghost'],            'anomaly_reason'] += 'ghost_dispatch, '
df.loc[df['rule_extreme_duration'], 'anomaly_reason'] += 'extreme_duration, '

if_only = (df['is_anomaly']==1) & (df['anomaly_reason']=='')
if_plus = (df['is_anomaly']==1) & (df['anomaly_reason']!='')
df.loc[if_only, 'anomaly_reason'] += 'IF_subtle, '
df.loc[if_plus, 'anomaly_reason'] += 'IF_confirmed, '

df['anomaly_reason'] = df['anomaly_reason'].str.rstrip(', ')
df.loc[df['anomaly_reason']=='', 'anomaly_reason'] = 'normal'

print(f"Total flagged (layered): {df['final_anomaly'].sum():,} ({df['final_anomaly'].mean()*100:.2f}%)")
print(f"\nBreakdown by reason:")
print(df[df['final_anomaly']==1]['anomaly_reason'].value_counts().head(15).to_string())

print(f"\nLayer contributions:")
print(f"  IF flagged:          {df['is_anomaly'].sum():,}")
print(f"  Rule instant:        {df['rule_instant'].sum():,}")
print(f"  Rule ghost:          {df['rule_ghost'].sum():,}")
print(f"  Rule impossible spd: {df['rule_impossible_speed'].sum():,}")
print(f"  Rule extreme dur:    {df['rule_extreme_duration'].sum():,}")

Total flagged (layered): 66,512 (1.47%)

Breakdown by reason:
anomaly_reason
IF_subtle                                         32763
instant_delivery                                  12348
impossible_speed, IF_confirmed                     9179
impossible_speed                                   3839
ghost_dispatch                                     3287
extreme_duration, IF_confirmed                     2629
extreme_duration                                   1864
instant_delivery, IF_confirmed                      515
instant_delivery, ghost_dispatch                     37
instant_delivery, ghost_dispatch, IF_confirmed       30
ghost_dispatch, extreme_duration                     15
ghost_dispatch, extreme_duration, IF_confirmed        4
ghost_dispatch, IF_confirmed                          2

Layer contributions:
  IF flagged:          45,122
  Rule instant:        12,930
  Rule ghost:          3,375
  Rule impossible spd: 13,018
  Rule extreme dur:    4,512


In [ ]:
with mlflow.start_run(run_name="layered_anomaly_system"):
    mlflow.log_param("layers", "IsolationForest + 4 rules")
    mlflow.log_param("rules", "instant, ghost, impossible_speed, extreme_duration")
    mlflow.log_metric("total_anomalies", int(df['final_anomaly'].sum()))
    mlflow.log_metric("anomaly_rate", df['final_anomaly'].mean())
    mlflow.log_metric("if_subtle_only", 32763)
    mlflow.log_metric("rule_caught", int(df['final_anomaly'].sum() - df['is_anomaly'].sum()))
    mlflow.set_tags({"model": "layered_hybrid", "stage": "final"})

import joblib
from pathlib import Path
MODEL_DIR = Path("../../models"); MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(iso, MODEL_DIR / "delivery_isolation_forest.pkl")

anomaly_output = df[df['final_anomaly']==1][[
    'order_id','city','courier_id','ds','delivery_duration_min','distance_km',
    'implied_speed_kmh','accept_hour','anomaly_score','anomaly_reason'
]].copy()
anomaly_output.to_parquet("../../data/processed/delivery_anomalies.parquet", index=False)

print(f"Saved IF model → {MODEL_DIR/'delivery_isolation_forest.pkl'}")
print(f"Saved {len(anomaly_output):,} flagged anomalies → delivery_anomalies.parquet")
print(f"\nStep 5 complete.")

Saved IF model → ../../models/delivery_isolation_forest.pkl
Saved 66,512 flagged anomalies → delivery_anomalies.parquet

Step 5 complete.
